# JustBuild Foundation - IT Ticket Triage Agent

Twenty-five IT tickets, one triage policy, one routing table. This notebook runs the same **three-agent** design you drew on the board, now pointed at the service desk:

**Input -> Agent 1 (Ingest & Extract) -> Agent 2 (Judge & Prioritise) -> Agent 3 (Communicate) -> You (the human)**

- **Agent 1** reads each messy ticket and pulls out the category and how bad the impact is.
- **Agent 2** sets the priority, routes to the right team, and attaches the response target. The priority matrix and routing live in **code**, not in the model.
- **Agent 3** drafts a handover to the on-call lead. It **resolves nothing** - that stays your call.

Everything the notebook needs - the tickets, the triage policy, and the **OC Brain** - is pulled from the hosted pack. Nothing is pasted inline.

> Run the cells top to bottom. Watch for the green checkmarks.

In [ ]:
#@title 1 · Setup  { display-mode: "form" }
!pip -q install google-generativeai pypdf requests pandas >/dev/null 2>&1
import requests, io, re, json, time
from pypdf import PdfReader
import pandas as pd
print('Setup complete - libraries ready.')

In [ ]:
#@title 2 · Configuration  { display-mode: "form" }
import getpass

MODE = "Live Gemini"  #@param ["Live Gemini", "Switch to OC Brain"]
MODEL = "gemini-2.5-flash-lite"  #@param {type:"string"}

BASE = "https://justbuild.orangecaterpillar.com/foundation-material/it-triage"
TICKETS_URL  = f"{BASE}/Tickets.pdf"
POLICY_URL   = f"{BASE}/Triage_Policy.pdf"
OC_BRAIN_URL = f"{BASE}/oc-brain.json"

GEMINI_API_KEY = ""
if MODE == "Live Gemini":
    GEMINI_API_KEY = getpass.getpass('Paste your Gemini API key (hidden). Leave blank to use the OC Brain: ').strip()

print(f'Mode: {MODE}   Model: {MODEL}')

In [ ]:
#@title 3 · Load the tickets, the policy, and the OC Brain  { display-mode: "form" }
def fetch_pdf_pages(url):
    r = requests.get(url, timeout=30); r.raise_for_status()
    reader = PdfReader(io.BytesIO(r.content))
    return [(p.extract_text() or '') for p in reader.pages]

ticket_pages = fetch_pdf_pages(TICKETS_URL)
TICKET_BLOCKS = []
for t in ticket_pages:
    if re.search(r'Ticket \d+ of \d+', t):
        TICKET_BLOCKS.append(re.sub(r'Ticket \d+ of \d+\s*', '', t).strip())

POLICY_TEXT = '\n'.join(fetch_pdf_pages(POLICY_URL))
OC_BRAIN = requests.get(OC_BRAIN_URL, timeout=30).json()

# Read the routing table and the SLA targets out of the policy. Fall back to the OC Brain's copy.
ROUTING = {}
for cat, team in re.findall(r'ROUTE:\s*(.+?)\s*\|\s*(.+)', POLICY_TEXT):
    if cat.strip().lower() != 'category':
        ROUTING[cat.strip()] = team.strip()
SLA = {}
for p, target in re.findall(r'SLA:\s*(P[1-4])\s*\|\s*(.+)', POLICY_TEXT):
    SLA[p] = target.strip()
if len(ROUTING) < 3:
    ROUTING = {r['category']: r['team'] for r in OC_BRAIN['routing_table']}
if len(SLA) < 3:
    SLA = {r['priority']: r['target'] for r in OC_BRAIN['sla_table']}

print(f'Loaded {len(TICKET_BLOCKS)} tickets, {len(ROUTING)} routes, {len(SLA)} SLA targets from the hosted pack.')
print(f"OC Brain on standby - {len(OC_BRAIN['tickets'])} prepared results loaded.")

In [ ]:
#@title 4 · The engine (and the OC Brain resilience layer)  { display-mode: "form" }
USE_OC_BRAIN = (MODE == 'Switch to OC Brain') or (not GEMINI_API_KEY)
if USE_OC_BRAIN and MODE == 'Live Gemini':
    print('No API key given - running on the OC Brain.')

model = None
if not USE_OC_BRAIN:
    import google.generativeai as genai
    genai.configure(api_key=GEMINI_API_KEY)
    model = genai.GenerativeModel(MODEL)

def call_gemini(prompt, tries=4):
    last = None
    for i in range(tries):
        try:
            return model.generate_content(prompt).text
        except Exception as e:
            last = e; msg = str(e)
            if '429' in msg or 'RESOURCE_EXHAUSTED' in msg: what = '429 (busy / free-tier limit)'
            elif '503' in msg or 'UNAVAILABLE' in msg:      what = '503 (model overloaded)'
            elif '404' in msg:                              what = '404 (model name)'
            else:                                           what = 'an error'
            wait = 2 ** i
            print(f'Gemini returned {what} - try {i+1}/{tries}. Waiting {wait}s...')
            time.sleep(wait)
    raise last

def extract_json(text):
    text = text.strip()
    text = re.sub(r'^```(?:json)?', '', text).strip()
    text = re.sub(r'```$', '', text).strip()
    start = min([i for i in (text.find('['), text.find('{')) if i != -1])
    return json.loads(text[start:])

def switch_to_oc_brain(reason):
    global USE_OC_BRAIN
    print(f'{reason} Switching to the OC Brain to keep going.')
    USE_OC_BRAIN = True

print('Engine ready.', '(OC Brain)' if USE_OC_BRAIN else f'(Live: {MODEL})')

## Agent 1 - Ingest & Extract
Turn 25 messy tickets into one clean table. Live, Gemini reads each ticket and judges category and impact; on the OC Brain, the same facts come from the prepared file.

In [ ]:
#@title 5 · Agent 1 runs  { display-mode: "form" }
CATEGORIES = list(ROUTING.keys())
records = []  # one row per ticket, in queue order

if not USE_OC_BRAIN:
    try:
        numbered = '\n\n'.join(f'--- TICKET {i+1} ---\n{t}' for i, t in enumerate(TICKET_BLOCKS))
        prompt = (
            'You are Agent 1 on an IT service desk. For EACH ticket below, read it and return ONLY a '
            'JSON array, in the same order (no prose), with exactly these fields:\n'
            '  ticket_id (string), requester (string), department (string),\n'
            '  category (one of: ' + ', '.join(CATEGORIES) + '),\n'
            '  scope ("single" one person, "team" a group/floor, or "company" everyone),\n'
            '  work_stopped (true if they cannot work at all with no workaround, else false),\n'
            '  security (true if it mentions phishing, malware, or a suspected breach, else false),\n'
            '  vip (true if the requester is an executive / their office, else false),\n'
            '  is_request (true if it is a routine request like new equipment, a software install, '
            'an access request, or a how-to, rather than something broken),\n'
            '  summary (a short phrase).\n\n' + numbered
        )
        for d in extract_json(call_gemini(prompt)):
            records.append({'ticket_id': d.get('ticket_id'), 'requester': d.get('requester'),
                            'extract': {k: d.get(k) for k in ('department','category','scope','work_stopped','security','vip','is_request','summary')}})
    except Exception as e:
        switch_to_oc_brain(f'The live model did not come through ({str(e)[:60]}).')

if USE_OC_BRAIN:
    records = []
    for tk in OC_BRAIN['tickets']:
        ex = tk['extract']
        records.append({'ticket_id': ex['ticket_id'], 'requester': ex['requester'],
                        'extract': {k: ex.get(k) for k in ('department','category','scope','work_stopped','security','vip','is_request','summary')}})

df1 = pd.DataFrame([{'Ticket': r['ticket_id'], 'Requester': r['requester'], 'Category': r['extract']['category'],
                     'Scope': r['extract']['scope'], 'Stopped': r['extract']['work_stopped'],
                     'Security': r['extract']['security'], 'VIP': r['extract']['vip'], 'Request': r['extract']['is_request']} for r in records])
print(f'Agent 1 extracted {len(df1)} tickets:')
df1

## Agent 2 - Judge & Prioritise  (rules in code)
Set priority, route to a team, attach the response target, and order the queue. The priority matrix, the routing, and the security **hard rule** all live in **code** - the policy decides, not the model's mood.

In [ ]:
#@title 6 · Agent 2 runs  { display-mode: "form" }
RANK = {'P1': 1, 'P2': 2, 'P3': 3, 'P4': 4}
SCOPE_SEV = {'company': 3, 'team': 2, 'single': 1}

def priority_of(ex):
    if ex.get('security'): return 'P1'                                  # hard rule
    if ex.get('is_request'): return 'P4'
    if ex.get('scope') == 'company': return 'P1'
    if ex.get('scope') == 'team' and ex.get('work_stopped'): return 'P1'
    if ex.get('scope') == 'team': return 'P2'
    if ex.get('work_stopped') and ex.get('vip'): return 'P1'
    if ex.get('work_stopped'): return 'P2'
    if ex.get('vip'): return 'P2'
    return 'P3'
def team_of(ex):
    if ex.get('security'): return 'Security Operations'                 # hard rule
    return ROUTING.get(ex.get('category'), 'Service Desk')
def evidence_of(ex):
    if ex.get('security'): return 'Security report; escalated to P1 and routed to Security Operations.'
    if ex.get('is_request'): return 'Routine service request.'
    if ex.get('scope') == 'company': return 'Company-wide outage.'
    if ex.get('scope') == 'team' and ex.get('work_stopped'): return 'A whole team cannot work.'
    if ex.get('scope') == 'team': return 'A team is affected; a workaround exists.'
    if ex.get('work_stopped') and ex.get('vip'): return 'VIP fully blocked.'
    if ex.get('work_stopped'): return 'User fully blocked, no workaround.'
    if ex.get('vip'): return 'VIP affected; a workaround exists.'
    return 'Single user, minor impact with a workaround.'

for r in records:
    ex = r['extract']
    if ex.get('scope') not in SCOPE_SEV: ex['scope'] = 'single'         # be forgiving of odd values
    r['priority'] = priority_of(ex); r['team'] = team_of(ex)
    r['sla'] = SLA.get(r['priority'], ''); r['evidence'] = evidence_of(ex)

# prioritise the queue: worst first, VIPs and wider impact ahead within a priority
queue = sorted(records, key=lambda r: (RANK[r['priority']], -(1 if r['extract'].get('vip') else 0),
                                       -SCOPE_SEV.get(r['extract'].get('scope'), 1)))

show = pd.DataFrame([{'#': i+1, 'Ticket': r['ticket_id'], 'Priority': r['priority'], 'Team': r['team'],
                      'SLA': r['sla'], 'Why': r['evidence']} for i, r in enumerate(queue)])
from collections import Counter
counts = Counter(r['priority'] for r in queue)
print('Agent 2 triaged the queue (rules applied in code):', dict(sorted(counts.items())))
show

In [ ]:
#@title    ...the P1s, front and centre  { display-mode: "form" }
P1 = [r for r in queue if r['priority'] == 'P1']
print(f'HANDLE NOW - {len(P1)} P1 ticket(s):\n')
for r in P1:
    print(f"  {r['ticket_id']}  {r['requester']}  ->  {r['team']}  [{r['sla']}]")
    print(f"     {r['evidence']}\n")

## Agent 3 - Communicate  (behind a human gate)
Draft a handover to the on-call IT lead. The agent **recommends**; it resolves nothing. You decide.

In [ ]:
#@title 7 · Agent 3 drafts (nothing is resolved)  { display-mode: "form" }
draft = None
if not USE_OC_BRAIN:
    try:
        summary = [{'ticket': r['ticket_id'], 'requester': r['requester'], 'priority': r['priority'],
                    'team': r['team'], 'sla': r['sla'], 'why': r['evidence']} for r in queue]
        prompt = (
            'You are Agent 3 on an IT service desk. Write a short, professional handover note to the '
            'on-call IT lead. List the P1 tickets first with their team and response target, then '
            'summarise the rest of the queue by priority. Make clear that nothing has been resolved or '
            'actioned and that assignment is theirs to confirm. Return only the note text.\n\n'
            + json.dumps(summary)
        )
        draft = call_gemini(prompt)
    except Exception as e:
        switch_to_oc_brain(f'The live model did not come through ({str(e)[:60]}).')
if USE_OC_BRAIN or draft is None:
    draft = OC_BRAIN['draft']

print('=' * 70)
print(draft)
print('=' * 70)
print('\nHUMAN GATE: this is a draft. The agent has resolved nothing and assigned nothing. You decide what happens next.')

In [ ]:
#@title 8 · Your decision  { display-mode: "form" }
DECISION = "Hold"  #@param ["Hold", "Approve the queue (I will assign it myself)", "Reject"]
if DECISION.startswith('Approve'):
    print('You approved the queue. The agent still assigns nothing - you dispatch it yourself, deliberately.')
elif DECISION == 'Reject':
    print('Rejected. Nothing leaves this notebook.')
else:
    print('On hold. Nothing is assigned. The decision stays with you.')

## Keep the result
Save the triaged queue and the handover to files you can download or paste into your notes, and compare the agent's queue with the one the room worked out by hand.

In [ ]:
#@title 9 · Save the triaged queue  { display-mode: "form" }
lines = ['# Triaged queue - IT Ops Ticket Triage', '',
         f'Mode: {"OC Brain" if USE_OC_BRAIN else MODEL}', '', '## Queue (worst first)', '']
for i, r in enumerate(queue, 1):
    lines.append(f"{i}. **{r['ticket_id']}** [{r['priority']}] - {r['requester']} -> {r['team']} ({r['sla']}) - {r['evidence']}")
lines += ['', '## Handover to the on-call lead', '', '```', draft, '```']
md_out = '\n'.join(lines)

with open('triage_queue.md', 'w') as f: f.write(md_out)
with open('triage_results.json', 'w') as f:
    json.dump({'mode': 'oc_brain' if USE_OC_BRAIN else MODEL,
               'queue': [{'ticket': r['ticket_id'], 'priority': r['priority'], 'team': r['team'],
                          'sla': r['sla'], 'why': r['evidence']} for r in queue],
               'draft': draft}, f, indent=2)

print('Saved: triage_queue.md and triage_results.json (open them from the file panel on the left).')
print('\n----- copy from here -----\n')
print(md_out)
# To download instead, uncomment:
# from google.colab import files; files.download('triage_queue.md')

## What you just saw
The same three narrow agents, chained: extract, judge, communicate - the judge is pure **code** again, because a priority policy has to be applied the same way every time, and the final call still belongs to a **human**. Swap this pack for your own and the pipeline serves a different job, which is exactly what you'll do next with your own use-case.